**Autor:** Andrés Alejandro Rodríguez Lozano  
**Portafolio:** Portafolio de Andrés Alejandro Rodríguez Lozano

# Capítulo 2 — Crecimiento del portafolio (US$10.000)

Rebalanceo anual vs `SPY`.

$$R_{p,t}=\sum_i w_{i,t} R_{i,t},\quad V_t=V_{t-1}(1+R_{p,t}),\quad V_0=10000$$


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%config InlineBackend.figure_formats = ['svg']
plt.rcParams.update({
    'figure.dpi': 72,
    'savefig.dpi': 72,
    'figure.figsize': (8, 3.8),
    'font.size': 9,
    'svg.fonttype': 'none',
    'path.simplify': True,
    'path.simplify_threshold': 0.2,
})

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import portfolio_utils as pu

def load_monthly():
    """Carga precios mensuales: cache mensual, diario, o yfinance."""
    mpath = ROOT / 'data' / 'precios_mensuales.csv'
    if mpath.exists():
        print('Precios mensuales (cache):', mpath.name)
        return pd.read_csv(mpath, index_col=0, parse_dates=True).loc[: '2026-09-30']
    csv = ROOT / 'data' / 'precios_ajustados_diarios.csv'
    if csv.exists():
        daily = pd.read_csv(csv, index_col=0, parse_dates=True)
        print('Precios diarios (cache):', csv.name)
        return pu.to_month_end(daily).loc[: '2026-09-30']
    tickers = sorted(set(list(pu.CURRENT_WEIGHTS) + [pu.BENCHMARK]))
    print('Descargando precios con yfinance…')
    daily = pu.download_prices(tickers)
    return pu.to_month_end(daily).loc[: '2026-09-30']


monthly = load_monthly()
bt = pu.backtest_portfolio(monthly, pu.CURRENT_WEIGHTS, start='2016-01-31', end='2026-09-30', rebalance='A')
spy = pu.backtest_portfolio(monthly, {'SPY': 1.0}, start='2016-01-31', end='2026-09-30', rebalance='N')
print('Final portafolio:', round(bt['value'].iloc[-1], 2))
print('Final SPY       :', round(spy['value'].iloc[-1], 2))


In [ ]:
# Gráfico de crecimiento — plt sobre series del backtest
fig, ax = plt.subplots()
ax.plot(bt.index, bt['value'], label='Portafolio', lw=1.8)
ax.plot(spy.index, spy['value'], label='SPY', lw=1.8, alpha=0.85)
ax.set_title('Crecimiento US$10.000 · rebalanceo anual · ene-2016 → sep-2026')
ax.set_ylabel('US$')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## Conclusiones
1. Los pesos actuales superan a SPY en valor terminal a sep-2026.

## Ejercicios
1. `rebalance='M'`.
2. Sin rebalanceo (`'N'`).
